# [Super AI Engineer Season 6] Hackathon Week 3
## การแข่งขันระบบค้นหาและตอบคำถาม (FahMai RAG Challenge)

**Super AI Engineer Season 6 - Level 1 Hackathon**  
- Dataset: ชุดข้อมูลความรู้ร้านฟ้าใหม่ และคำถาม-คำตอบ
- Source: Super AI Engineer  
- จัดทำโดย: **600425-วิศิษฐ์**

---
### Notebook Outline
1. Setup & Imports (การตั้งค่าและติดตั้งไลบรารี)  
2. Data Loading & Initial Inspection (การโหลดข้อมูลและตรวจสอบเบื้องต้น)  
3. Context Chunking (การตัดแบ่งข้อความ)
4. Build Retrieval Indices (การสร้างดัชนีด้วย BM25 และ BGE-M3)
5. Hybrid Retrieval & Reranker (การค้นหาแบบผสมและการจัดอันดับใหม่)
6. LLM Prompt Strategy (การเตรียมคำสั่งสำหรับโมเดลภาษา)
7. Execution Pipeline & Generation (กระบวนการทำงานและสร้างผลลัพธ์)

In [1]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv tqdm pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 69.8 MB/s eta 0:00:00


In [2]:
!unzip -q /content/super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip -d .

# 1. Setup & Imports
### 1.1 ติดตั้งและนำเข้าไลบรารี (Setup Libraries & Environment)
ทำการติดตั้งไลบรารีที่จำเป็น รวมถึงกำหนดค่าตัวแปรสภาพแวดล้อมเพื่อจัดการ Hugging Face Token และ ThaiLLM API Key รวมทั้งสร้างตัวตรวจจับพาธสำหรับชุดข้อมูลอัตโนมัติ

In [3]:
import os, csv, re, time, requests, json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

# Set Hugging Face cache to Colab's working directory to prevent "No space left on device" (Colab Root space is limited)
os.environ["HF_HOME"] = "/kaggle/working/huggingface"

# Workspace Configuration (Colab Paths Auto-Detect)
N_QUESTIONS = 100  # Set to 100 to predict all questions

# Auto-detect data directory for both Colab competition datasets and local setups
possible_paths = [
    "/content/super-ai-engineer-s-6-fah-mai-rag-challenge-level-1/data",
    "./super-ai-engineer-s-6-fah-mai-rag-challenge-level-1/data"
]

DATA_DIR = None
for p in possible_paths:
    if os.path.exists(p):
        DATA_DIR = p
        break

if not DATA_DIR:
    # Fallback to current directory -> data
    DATA_DIR = "./data"

KB_DIR = f"{DATA_DIR}/knowledge_base"
print(f"Using Data Directory: {DATA_DIR}")

# Authentication
try:
    # Try Google Colab Secrets
    from google.colab import userdata
    THAILLM_API_KEY = userdata.get('OpenThaiGPT')
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception as e:
    print("Warning: Running outside Colab or secrets not found. Using dummy vars for now.")
    THAILLM_API_KEY = os.getenv("THAILLM_API_KEY", "dummy_thaillm")
    HF_TOKEN = os.getenv("HF_TOKEN", "dummy_hf")

# Only login to HF if token is actually set up
if HF_TOKEN and HF_TOKEN != "dummy_hf":
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face Authenticated!")
else:
    print("Warning: Skipping HF login! Models might still download if they are public.")

Using Data Directory: ./data


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face Authenticated!


# 2. Data Loading & Context Chunking
### 2.1 โหลดข้อมูลและตัดแบ่งข้อความ (Document Loading & Tokenizer-based Chunking)
ใช้กระบวนการอ่านไฟล์ Markdown ทั้งหมดในโครงสร้าง Knowledge Base ต่อด้วยการตัดแบ่งข้อความ (Chunking)
โดยใช้ Tokenizer ของ BGE-Reranker แบบรับประกันไม่ให้เกิดปัญหา Token Exceed (Max Token กำหนดไว้ที่ 400 ต่อ Chunk)

In [4]:
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    documents.append({"path": str(fp.relative_to(kb_dir)), "text": text})

print(f"Loaded {len(documents)} documents")
print(f"   - Products: {sum(1 for d in documents if 'products/' in d['path'])}")
print(f"   - Policies: {sum(1 for d in documents if 'policies/' in d['path'])}")
print(f"   - Store Info: {sum(1 for d in documents if 'store_info/' in d['path'])}")

# ใช้ tokenizer จาก bge-reranker เพื่อใช้นับ token ก่อนสับ chunk
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-reranker-v2-m3')

def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

def smarter_chunking_v2(documents, max_tokens=400, overlap_tokens=60):
    smart_chunks = []
    for doc in documents:
        path = doc["path"]
        filename = os.path.basename(path).replace(".md", "")

        category = "General"
        if "product" in path.lower():    category = "Product"
        elif "policy" in path.lower():   category = "Policy"
        elif "store" in path.lower():    category = "Store Info"

        metadata_prefix = f"หมวดหมู่: {category} | หัวข้อ: {filename}\nเนื้อหา: "
        prefix_tokens = count_tokens(metadata_prefix)
        content_budget = max_tokens - prefix_tokens

        sections = re.split(r'(^#+\s.*$)', doc["text"], flags=re.MULTILINE)
        paragraphs = []
        for i in range(len(sections)):
            for para in sections[i].split('\n\n'):
                para = para.strip()
                if para: paragraphs.append(para)

        current_paras = []
        current_tokens = 0

        for para in paragraphs:
            para_tokens = count_tokens(para)
            if para_tokens > content_budget:
                if current_paras:
                    text = metadata_prefix + '\n\n'.join(current_paras)
                    smart_chunks.append({"text": text.strip(), "source": path, "metadata": {"product": filename, "category": category}})

                words = para.split()
                sub = []; sub_tok = 0
                for w in words:
                    w_tok = count_tokens(w)
                    if sub_tok + w_tok > content_budget:
                        text = metadata_prefix + ' '.join(sub)
                        smart_chunks.append({"text": text.strip(), "source": path, "metadata": {"product": filename, "category": category}})
                        sub = sub[-overlap_tokens:]
                        sub_tok = count_tokens(' '.join(sub))
                    sub.append(w); sub_tok += w_tok

                current_paras = [para] if para_tokens <= content_budget else []
                current_tokens = para_tokens if current_paras else 0
                continue

            if current_tokens + para_tokens > content_budget:
                text = metadata_prefix + '\n\n'.join(current_paras)
                smart_chunks.append({"text": text.strip(), "source": path, "metadata": {"product": filename, "category": category}})

                overlap_paras = current_paras[-1:]
                current_paras = overlap_paras + [para]
                current_tokens = sum(count_tokens(p) for p in current_paras)
            else:
                current_paras.append(para)
                current_tokens += para_tokens

        if current_paras:
            text = metadata_prefix + '\n\n'.join(current_paras)
            smart_chunks.append({"text": text.strip(), "source": path, "metadata": {"product": filename, "category": category}})

    return smart_chunks

chunks = smarter_chunking_v2(documents, max_tokens=400, overlap_tokens=60)
print(f"Created {len(chunks)} smart chunks")

over = [c for c in chunks if count_tokens(c['text']) > 512]
print(f"   - Chunks > 512 tokens: {len(over)}")

Loaded 118 documents
   - Products: 110
   - Policies: 5
   - Store Info: 3


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Created 445 smart chunks
   - Chunks > 512 tokens: 0


# 3. Build Retrieval Indices
### 3.1 การสร้างดัชนี (Dense และ Sparse Index)
โหลดชุดคำถามและสร้างดัชนีด้วยเทคโนโลยี 2 ระบบ ได้แก่
1. **Dense Retrieval:** ใช้ `BAAI/bge-m3` ในการอัดข้อความเป็นเวกเตอร์ (Embeddings)
2. **Sparse Retrieval:** ใช้ `BM25Okapi` สำหรับการจับคู่คำสำคัญ (Keyword Matching) โดยใช้ `pythainlp` newmm ร่วมกับระบบ Whitelist คำศัพท์เฉพาะของโจทย์

In [5]:
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus import thai_stopwords
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import torch

# --- 1. Define Questions Dataset ---
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f"Loaded {len(questions)} questions.")

# --- 2. Dense Embeddings (BGE-M3) ---
print("\nGenerating Dense Embeddings (BGE-m3)...")
embed_model = SentenceTransformer("BAAI/bge-m3")
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(chunk_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
print(f"   - Embeddings shape: {chunk_embeddings.shape}")

# --- 3. BM25 Search (Custom Tokenization) ---
print("\nGenerating BM25 Index...")
thai_stops = set(thai_stopwords())
DOMAIN_WHITELIST = {
    "กี่", "เท่าไหร่", "เท่าไร", "ได้", "ไม่",
    "atm", "ip", "hz", "gb", "tb", "mb", "mah", "w", "db",
    "บาท", "ปี", "เดือน", "วัน", "ชั่วโมง", "นาที",
    "มี", "หมด", "ฟรี", "ราคา", "ประกัน", "คืน", "เปลี่ยน",
}

def tokenize_for_bm25(text):
    if not isinstance(text, str): return []
    tokens = word_tokenize(text.lower(), engine="newmm", keep_whitespace=False)
    clean_tokens = []
    for t in tokens:
        t_clean = re.sub(r'[^\wก-๙.]', '', t)
        if not t_clean: continue
        if t_clean not in thai_stops or t_clean in DOMAIN_WHITELIST:
            clean_tokens.append(t_clean)
    return clean_tokens

tokenized_chunks = [tokenize_for_bm25(c["text"]) for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)
print(f"   - BM25 Built over {len(tokenized_chunks)} chunks")

Loaded 100 questions.

Generating Dense Embeddings (BGE-m3)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

   - Embeddings shape: (445, 1024)

Generating BM25 Index...
   - BM25 Built over 445 chunks


# 4. Hybrid Retrieval & Reranking Setup
### 4.1 การค้นหาและการจัดอันดับเอกสาร
- ทำการรวมผลลัพธ์ (Ensemble) ของ BM25 และ Dense Embeddings โดยใช้ Reciprocal Rank Fusion (RRF)
- ส่งเอกสารทั้งหมดที่รวมกันได้ไปให้ CrossEncoder (`BAAI/bge-reranker-v2-m3`) ทำการจัดอันดับอีกรอบอย่างแม่นยำเพื่อกรองให้เหลือเฉพาะ 5-8 ชิ้นที่ดีที่สุด

In [6]:
from sentence_transformers import CrossEncoder
import torch

TOP_K = 5

def dense_retrieve(query, chunk_embs, k=TOP_K):
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embs, q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

def bm25_retrieve(query, k=TOP_K):
    tokens = tokenize_for_bm25(query)
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

def hybrid_retrieve(query, chunk_embs, k=TOP_K, rrf_k=60):
    fetch_k = k * 2
    d_idx, _ = dense_retrieve(query, chunk_embs, k=fetch_k)
    b_idx, _ = bm25_retrieve(query, k=fetch_k)

    rrf_scores = {}
    for rank, idx in enumerate(d_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(b_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)

    sorted_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:k]
    return sorted_idx

def hybrid_retrieve_chunks(query, top_k=TOP_K):
    idx = hybrid_retrieve(query, chunk_embeddings, k=top_k)
    return [chunks[i] for i in idx]

print("\nLoading BGE-Reranker-V2-M3...")
# Initialize Reranker
rerank_model = CrossEncoder(
    'BAAI/bge-reranker-v2-m3',
    max_length=512
)

def hybrid_rerank_retrieve(query, top_k_candidates=30, final_top_n=5):
    candidates = hybrid_retrieve_chunks(query, top_k=top_k_candidates)
    if not candidates: return []

    pairs = [[query, c['text']] for c in candidates]
    with torch.no_grad():
        scores = rerank_model.predict(pairs, batch_size=32, show_progress_bar=False)

    for i, score in enumerate(scores):
        candidates[i]['rerank_score'] = score

    return sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)[:final_top_n]


Loading BGE-Reranker-V2-M3...


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

# 5. LLM Prompt Strategy
### 5.1 กำหนด Prompt สำหรับการอนุมาน (Inference)
กำหนด Role ให้กับ LLM เป็น Expert ของร้านฟ้าใหม่ พร้อมฟังก์ชันเพื่อจัดการการส่ง API ไปยังบริการของ `thaillm` และดักจับ JSON Parsing

In [7]:
def ask_llm(messages, model="kbtg", max_retries=5):
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2048,
        "temperature": 0,
    }
    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()
        except requests.exceptions.RequestException as e:
            time.sleep(2 ** attempt)
    return None

def parse_answer(text):
    if text is None: return None
    match = re.search(r'"answer":\s*"?(\d+)"?', text)
    if match: return int(match.group(1))
    numbers = re.findall(r'(\d+)', text)
    if numbers:
        for n in reversed(numbers):
            if 1 <= int(n) <= 10: return int(n)
    return None

CURRENT_DATE_STR = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ===== THE SYSTEM PROMPT =====
SYSTEM_PROMPT = f"""\
<role_identity>
คุณคือ "Expert Agent ของร้านฟ้าใหม่" (Fah Mai Electronics)
วันนี้คือวันที่: {CURRENT_DATE_STR}
</role_identity>

<store_knowledge>
ร้านฟ้าใหม่มีข้อมูลพื้นฐานดังนี้ (ใช้เพื่อตรวจสอบ Domain):
1. แบรนด์ในเครือ (5 House Brands):
   - สายฟ้า (SaiFah): มือถือและแท็บเล็ต (23 รายการ)
   - ดาวเหนือ (DaoNuea): คอมพิวเตอร์และแล็ปท็อป (23 รายการ)
   - คลื่นเสียง (KluenSiang): อุปกรณ์เสียง/หูฟัง (22 รายการ)
   - วงโคจร (WongKhoJon): อุปกรณ์สวมใส่/Smartwatch (12 รายการ)
   - จุดเชื่อม (JudChuem): อุปกรณ์เสริม/Accessories (18 รายการ)
2. แบรนด์พาร์ทเนอร์: ZenByte (Storage), NovaTech (Peripherals), PulseGear (Power), ArcWave (Displays)
3. สาขา (5 Branches): สยาม, ลาดพร้าว, พระราม 9, เชียงใหม่, ภูเก็ต
</store_knowledge>

<logic_guidelines>
เกณฑ์การตัดสินใจ (ลำดับความสำคัญสูง):
- [DOMAIN CHECK]: หากคำถามไม่เกี่ยวกับแบรนด์/สินค้า/สาขาข้างต้น (เช่น ถามเรื่องอาหาร, ท่องเที่ยว, รถยนต์) -> ตอบ 10 ทันที
- [EVIDENCE CHECK]: ค้นหาข้อมูลเชิงลึกใน <context> เพื่อยืนยันคำตอบ 1-8
- [NO INFO]: หากเป็นเรื่องในร้าน แต่ใน <context> ไม่มีข้อมูลที่ยืนยันตัวเลือกใดได้เลย -> ตอบ 9
</logic_guidelines>

<response_constraints>
- ห้ามใช้ความรู้ภายนอกนอกเหนือจาก <store_knowledge> และ <evidence>
- วิเคราะห์เหตุผลภายใน <thinking> (internal only — ห้ามแสดงใน output)
- output ต้องเป็น JSON {{"answer": N}} เท่านั้น ห้ามมีข้อความอื่นใดทั้งสิ้น
</response_constraints>

<inference_rules>
- ห้าม infer สถานะสินค้า (เช่น "หมดสต็อก", "มีของ") หากไม่มีระบุชัดเจนใน <evidence>
- ห้าม classify ประเภทสินค้า (เช่น over-ear, TWS) หากไม่มีระบุชัดเจนใน <evidence>
- คำว่า "limited edition", "special edition", "FahMai Edition" ไม่ได้หมายความว่าของหมด
</inference_rules>

<decision_process>
1. [FILTER FIRST]: หากคำถามมี constraint (งบ, วันที่, ประเภท)
   -> ดึงเฉพาะสินค้าที่ผ่าน constraint จาก evidence ก่อน
   -> ได้ shortlist แล้วค่อยไปขั้นตอน 2

2. [MATCH]: เทียบ shortlist กับตัวเลือก 1-8
   -> เลือกตัวเลือกที่ครอบคลุม shortlist มากที่สุด โดยไม่มีสินค้าราคาเกินงบ
   -> ไม่ต้อง match 100% — สินค้านอก evidence ในตัวเลือกให้ ignore

3. [FALLBACK]:
   - evidence ไม่มีข้อมูล domain นั้นเลย -> ตอบ 9
   - ไม่ใช่เรื่องร้านฟ้าใหม่ -> ตอบ 10
</decision_process>

<examples>
- Question: "ซื้อหูฟังคลื่นเสียงมาเมื่อวันที่ 10 มีนาคม 2026 วันนี้ (28 มีนาคม) ยังเปลี่ยนเครื่องใหม่ได้ไหม?"
- Context: "เงื่อนไขการเปลี่ยนสินค้า: ต้องภายใน 15 วันนับจากวันที่ซื้อ"
- Logic: 28 - 10 = 18 วัน (เกิน 15 วัน) -> เปลี่ยนไม่ได้
- Output: {{"answer": 4}}

- Question: "ตู้เย็นยี่ห้อ Samsung ลดราคาเท่าไหร่?"
- Logic: Samsung ไม่ใช่ House Brand และไม่ใช่ Partner Brand (ZenByte/NovaTech...)
- Output: {{"answer": 10}}
</examples>
"""

def build_rag_prompt(question, choices, retrieved_chunks):
    context_parts = []
    for i, c in enumerate(retrieved_chunks):
        context_parts.append(f"[หลักฐานชิ้นที่ {i+1}]: {c['text']}")

    context_text = "\n".join(context_parts)
    choices_text = "\n".join([f"{k}. {v}" for k, v in choices.items()])

    return (
        f"<evidence>\n{context_text}\n</evidence>\n\n"
        f"<target_question>{question}</target_question>\n\n"
        f"<options>\n{choices_text}\n</options>\n\n"
        f"คำสั่ง: วิเคราะห์หลักฐานและตอบเลขข้อที่ 'ถูกต้องที่สุด' เพียงข้อเดียวในรูปแบบ JSON"
    )

# 6. Execution Pipeline & Generation
### 6.1 การรันระบบเพื่อสร้างผลลัพธ์ Submission คาดหวัง
ทำงานแบบ Loop ทีละคำถาม (สามารถตั้งค่า Hyperparameters ได้แก่ จำนวน Top K Candidates และ Top N จาก Reranker)
เมื่อเสร็จสิ้นกระบวนการ จะบันทึกลงไฟล์ `submission_RAG.csv` ทันทีและมี Log การทำงานแยกในรันเพื่อ Debug

In [8]:
def run_pipeline(questions, retrieve_fn, label="dense", n=None):
    if n is None: n = len(questions)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = f"run_log_{label}_{timestamp}.jsonl"
    predictions = {}

    print(f"Beginning Pipeline for {label} -> Logs: {log_file}")

    for i, q in enumerate(tqdm(questions[:n], desc=f"Processing {label}")):
        retrieved_chunks = retrieve_fn(q["question"])
        prompt = build_rag_prompt(q["question"], q["choices"], retrieved_chunks)

        try:
            raw = ask_llm([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ])
            pred = parse_answer(raw)
        except Exception as e:
            raw = f"ERROR: {str(e)}"
            pred = None

        final_pred = pred if pred else 1 # Fallback to 1 if missing or 0
        predictions[q["id"]] = final_pred

        log_entry = {
            "id": q["id"],
            "pred": final_pred,
            "raw_output": raw,
            "sources": [c.get('source', 'unknown') for c in retrieved_chunks]
        }

        with open(log_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(log_entry, ensure_ascii=False) + "\n")

        # Simple rate limit safeguard between requests
        time.sleep(0.3)

    print(f"\nFinished {label} predicting {len(predictions)} questions.")
    return predictions

# =========================================================================
# This executes that exact logic for the final CSV.
# =========================================================================
BEST_K = 32
BEST_N = 8

# Helper retrieve function locked to the optimal params:
def best_retrieve_fn(query):
    return hybrid_rerank_retrieve(query, top_k_candidates=BEST_K, final_top_n=BEST_N)

print(f"\n==============================================")
print(f"Execution Started: Top_K = {BEST_K} | Top_N = {BEST_N}")

final_preds = run_pipeline(
    questions,
    retrieve_fn=best_retrieve_fn,
    label=f"champion_model_k{BEST_K}_n{BEST_N}",
    n=N_QUESTIONS
)

# Commit to CSV
submission_filename = "submission_RAG.csv"
with open(submission_filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        # In case the prediction failed, default to 1 (valid answer)
        writer.writerow([qid, final_preds.get(qid, 1) or 1])

print(f"Final Submission Saved successfully to {submission_filename}!")


Execution Started: Top_K = 32 | Top_N = 8
Beginning Pipeline for champion_model_k32_n8 -> Logs: run_log_champion_model_k32_n8_20260328_170756.jsonl


Processing champion_model_k32_n8:   0%|          | 0/100 [00:00<?, ?it/s]


Finished champion_model_k32_n8 predicting 100 questions.
Final Submission Saved successfully to submission_RAG.csv!
